# Week 3 Day 3 (free route) - The context layer and memory

## Running UnderCover's context layer on your laptop

On the AWS route, today's lab puts the context layer on Lambda as its own MCP server, behind the AgentCore Gateway with the app, and gives the agent an AgentCore Memory. On the free route the context layer runs on your laptop beside the agent, the agent calls it and the app directly, and the agent works without a memory.

The Gateway and the Memory are AWS services, so for those you can watch me. Everything else in today's build works the same way on your laptop: the agent looks things up in the context layer, each lookup says where it came from, and the decision record holds its calculation.

This notebook has no code to run. It gives you the steps.

## 4 steps:

STEP 1: Update the app on Render

STEP 2: Look at the context layer

STEP 3: Run a returning customer's claims

STEP 4: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's lab. The main lab, <a href="week3day3.ipynb">week3day3.ipynb</a>, runs on AWS. You need the Render service from Day 2, and today's build of the product, yours or <code>week3day3-end</code>.<br/><br/>Step 3 works two claims, which uses part of your 50 free requests for the day. If you run out, the requests come back the next day.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app on Render

Today's build changes the app: the documents moved to the context layer, and the decision record checks itself against the claims manual. So the app on Render needs today's code.

- **To deploy my build,** open your web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day3-end`, and choose **Save changes**. Render starts the deploy straight away.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2, and choose **Manual Deploy** and **Deploy latest commit**.

The build and start commands and the environment variables stay as they were on Day 2. The context layer doesn't go on Render: it runs on your laptop beside the agent, which is the only thing that calls it.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Look at the context layer

Open the `context` folder in the `undercover` repository. It holds seven data files, each standing in for another system at the firm or an outside provider, and each naming its source system, its owner and when it was last reviewed:

| File | What it holds | Where it would come from |
|---|---|---|
| `data_dictionary.yaml` | The meaning of each field the agent sees | The data catalog |
| `glossary.yaml` | The terms the policy wording defines | The policy wording |
| `plans.yaml` | Each plan's limits, waiting periods and deductible | Policy administration |
| `claim_rules.yaml` | The filing window, the look-back, repeat claims and meals | Claims operations |
| `reference_data.yaml` | Outside events, places and products | Outside data providers |
| `policy_wording.md` | The contract with the customer, by section | Legal |
| `claims_manual.md` | How handlers work a claim, by section | Claims operations |

`npm run dev` now starts the context layer's MCP server beside the backend, the agent's service and the dashboard, at `http://127.0.0.1:8006/mcp`. When there's no Gateway in `.env`, the agent calls it there directly, and calls the app directly with its API key, as it did before today.

## Step 3: Run a returning customer's claims

Oliver Grant (CUS-3005) has two claims in the queue. In July a giraffe ate his passport in Nairobi, and in August a coconut fell on him in Sri Lanka. His second claim opens "Yes, it is me again".

In a terminal in the `undercover` folder, reseed so both claims are waiting, then start everything:

```
npm run seed
npm run dev
```

In a second terminal, work the two claims, one after the other:

```
uv run -m agent CLM-1005
uv run -m agent CLM-1006
```

Open each claim's page on the dashboard, at `http://localhost:5173/claims/CLM-1005` and `http://localhost:5173/claims/CLM-1006`, or on your app on Render, signed in as Dana Whitfield. Both read the same Supabase database. Look for:

- the lookups in the context layer, each saying which file it came from and who owns it;
- the Context used panel, with everything the agent looked up, grouped by file;
- the decision record, with the calculation line by line, who made the decision and the claims manual's version.

There is no memory step on CLM-1006. Without a memory, the agent knows only what our records say about Oliver, and his claim history in the records is empty.

## Step 4: Compare with the AWS route

In the main lab, the same two claims run with the Gateway and the Memory. Watch Part 5 there: once the agent has decided CLM-1005, AgentCore turns the claim and the decision into facts about Oliver and a summary of the claim, and on CLM-1006 the agent can search them and find the giraffe.

There's nothing to tear down. Your app on Render costs nothing, so keep it for Day 4.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: give the agent a memory of its own without AWS. LangGraph's <code>PostgresStore</code> can keep it in your Supabase database, with pgvector to search it, in a namespace for each customer like the AWS route's. Save the claim and the decision when the agent has decided, and give the agent a tool to search its customer's memories.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Local verification of the Render (free route) deployment.
import os
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
APP = os.getenv("UNDERCOVER_API", "https://undercover-ap.onrender.com").rstrip("/")
print("Checked at (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("App:", APP)


def check(path):
    started = time.time()
    try:
        response = requests.get(APP + path, timeout=120)
        body = response.text.strip().replace("\n", " ")[:160]
        return f"{path}: HTTP {response.status_code} in {time.time() - started:.1f}s - {body}"
    except Exception as exc:
        return f"{path}: {type(exc).__name__}: {str(exc)[:160]}"


for path in ["/api/health", "/sign-in", "/api/claims/CLM-1901"]:
    print(check(path))

for name in ["UNDERCOVER_AGENT", "UNDERCOVER_COPILOT"]:
    print(f"{name} setting:", os.getenv(name, "not set locally (set in Render dashboard)"))


Checked at (UTC): 2026-11-27T05:31:44+00:00
App: https://undercover-ap.onrender.com


/api/health: HTTP 200 in 0.5s - {"status":"ok"}


/sign-in: HTTP 200 in 0.5s - <!doctype html> <html lang="en">   <head>     <meta charset="UTF-8" />     <meta name="viewport" content="width=device-width, initial-scale=1.0" />     <title>U


/api/claims/CLM-1901: HTTP 401 in 0.7s - {"detail":"Sign in, or send the agent's API key"}
UNDERCOVER_AGENT setting: not set locally (set in Render dashboard)
UNDERCOVER_COPILOT setting: not set locally (set in Render dashboard)


## Solution - assignment challenge

The free route has no AgentCore Memory, but LangGraph's `PostgresStore` can keep the same kind of memory in the Supabase database we already use for checkpoints. We give it `index={"dims": 1536, "embed": ..., "fields": ["text"]}` so `setup()` creates a pgvector column and index, and we embed with `openai/text-embedding-3-small` through OpenRouter (1,536 dimensions). Supabase keeps pgvector in the `extensions` schema, so the connection's search path is `lab,extensions`.

Memories live in one namespace per customer, `("customers", customer_id)`, the equivalent of the AWS route's `/customers/{actorId}`. When the agent records a decision, a graph node saves two items under that namespace: the claim (summary, benefit, date, place and items, but **not** the customer's free text or documents) and the decision (payout, section and reason). Referred claims are not saved, so an injected instruction can't be carried into a customer's later claims. The `search_memory(query)` tool is built for one customer and searches only that namespace.

The last cell saves the two Oliver Grant claims and searches for the giraffe with a tool bound to CUS-3005. It checks that one customer's memory works and is scoped correctly.

**None of the cells below were executed in this notebook** (they need `DATABASE_URL`, an OpenRouter key for embeddings, and the `vector` extension in Supabase). They are complete, runnable code against the variables defined above.

In [ ]:
# Connect a LangGraph PostgresStore to Supabase, with pgvector search.
import json
from pathlib import Path

import psycopg
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from langgraph.store.postgres import PostgresStore
from psycopg.rows import dict_row

load_dotenv("../.env", override=True)
DATABASE_URL = os.getenv("DATABASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Supabase puts pgvector in the extensions schema; LangGraph's migrations
# need to see the `vector` type on the search path.
memory_connection = psycopg.connect(
    DATABASE_URL, autocommit=True, prepare_threshold=0, row_factory=dict_row,
    options="-c search_path=lab,extensions,public",
)
memory_connection.execute("CREATE SCHEMA IF NOT EXISTS lab")
memory_connection.execute("CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA extensions")

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
)

store = PostgresStore(
    memory_connection,
    index={"dims": 1536, "embed": embeddings, "fields": ["text"]},
)
store.setup()
print("PostgresStore tables and the pgvector index are ready in Supabase")

In [ ]:
# One namespace per customer: no customer's memories reach another's claim.
def customer_namespace(customer_id):
    """The memory namespace for one customer."""
    return ("customers", customer_id)


def remember_claim_and_decision(claim, decision):
    """Save the claim and the decision once the agent has decided."""
    namespace = customer_namespace(claim["customer_id"])
    claim_id = claim["claim_id"]
    items = ", ".join(
        f"{item['description']} (${item['amount']:,.2f})" for item in claim.get("items", [])
    )
    claim_text = (
        f"Claim {claim_id}: {claim['summary']}. Benefit: {claim.get('claim_type')}. "
        f"Incident on {claim.get('incident_date')} in {claim.get('incident_location')}. "
        f"Items: {items}."
    )
    decision_text = (
        f"Decision on {claim_id}: {decision['decision']}, "
        f"payout ${decision.get('payout', 0):,.2f}. "
        f"Section: {decision.get('reference', '')}. Reason: {decision.get('rationale', '')}."
    )
    store.put(namespace, f"{claim_id}:claim", {"text": claim_text, "claim_id": claim_id})
    store.put(namespace, f"{claim_id}:decision", {"text": decision_text, "claim_id": claim_id})
    print(f"Saved two memories under {namespace}")


def make_memory_saver(claims_by_id):
    """A graph node to add after `decide`, so saves survive a restart."""
    def save_memory(state):
        """Save a decided claim; never save a referral or a screened claim."""
        claim = claims_by_id[state["claim_id"]]
        if state["decision"] == "refer":
            return None
        remember_claim_and_decision(claim, state)
        return None
    return save_memory


def make_search_memory_tool(customer_id):
    """Build a memory tool that can only search this customer's namespace."""
    namespace = customer_namespace(customer_id)

    @tool
    def search_memory(query: str) -> str:
        """Search what we remember about this customer from their earlier claims."""
        found = store.search(namespace, query=query, limit=5)
        if not found:
            return "Nothing remembered about this customer yet."
        return "\n\n".join(f"[{item.key}] {item.value['text']}" for item in found)

    return search_memory

In [ ]:
# Save Oliver Grant's two claims and search his memory for the giraffe.
def read_json(name):
    """Read one of UnderCover's JSON data files."""
    return json.loads((Path("../data") / name).read_text(encoding="utf-8"))


claims_by_id = {claim["claim_id"]: claim for claim in read_json("claims.json")}
decisions_by_id = {row["claim_id"]: row for row in read_json("evals/decisions.json")}

for claim_id in ["CLM-1005", "CLM-1006"]:
    claim = claims_by_id[claim_id]
    handler = decisions_by_id[claim_id]
    remember_claim_and_decision(claim, {
        "decision": handler["decision"],
        "payout": handler["payout"],
        "reference": handler["reference"],
        "rationale": handler["reason"],
    })

search_memory = make_search_memory_tool("CUS-3005")
print(search_memory.invoke({"query": "giraffe ate the passport in Nairobi"}))

# Give this tool to the claim agent when it works CUS-3005's claims:
# tools=[..., make_search_memory_tool(claim["customer_id"])]
# and add make_memory_saver(claims_by_id) as a node after `decide`.